<a href="https://colab.research.google.com/github/john-dechellis-weather/wx_compare/blob/main/wx_compare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
!apt-get install -y libeccodes0 > /dev/null 2>&1
!pip install -q cfgrib xarray eccodes requests pandas matplotlib

In [61]:
import os, sys

REPO_URL = "https://github.com/john-dechellis-weather/wx_compare.git"
REPO_DIR = "/content/wx_compare"

os.chdir('/content')
if os.path.exists(REPO_DIR):
    !rm -rf {REPO_DIR}
!git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

Cloning into '/content/wx_compare'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 50 (delta 11), reused 8 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 26.17 KiB | 3.74 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [62]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
CACHE_ROOT = Path('/content/drive/MyDrive/wx_compare_cache')
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from datetime import datetime, timezone, timedelta
from models import GfsMos, Hrrr, Station
from compare import run_comparison

CYCLE = (datetime.now(timezone.utc) - timedelta(days=1)).replace(
    hour=12, minute=0, second=0, microsecond=0)

STATIONS_META = [
    Station(icao='KJFK', lat=40.6398, lon=-73.7789, elev_ft=13.0),
]
STATION_IDS = [s.icao for s in STATIONS_META]

sources = [
    GfsMos(cache_dir=CACHE_ROOT / 'gfs_mos'),
    Hrrr(cache_dir=CACHE_ROOT / 'hrrr',
         stations=STATIONS_META,
         fhours=range(0, 19)),
]

df = run_comparison(sources, CYCLE, STATION_IDS)
df

In [ ]:
from compare import plot_comparison
plot_comparison(df, 'KJFK')